# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjalanano-dev/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane:** Refresh / Content Opportunity Scoring (Lane 2)  
**Dataset:** `data/raw/content_refresh_anonymized.csv` (30,000 rows x 44 columns)  
**AI assistant note:** skills loaded — `building-baselines/SKILL.md` + `flyrank/flyrank-data/SKILL.md`

> This notebook does three things in order:
> 1. Checks two signals with bucket tables and verdicts
> 2. Encodes ONE rule: a score, ONE reason code, an action label, writes the ranked CSV
> 3. Reads the top 10 with a skeptic's eye

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import os, json

# Resolve the data path: works locally and in Colab
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
DATA_PATH = os.path.join(REPO_ROOT, 'data', 'raw', 'content_refresh_anonymized.csv')
if not os.path.exists(DATA_PATH):
    DATA_PATH = '/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(DATA_PATH)

print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Clients: {df["client_id"].nunique()}')
print(f'Content types: {df["content_type"].value_counts().to_dict()}')

# Build evaluation label from trend_direction
# IMPORTANT: trend_direction and trend_pct are label sources -- NEVER features
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
base_rate = df['is_declining_label'].mean()
print(f'\nBase rate of decline (is_declining_label): {base_rate:.1%}')

Loaded: 30,000 rows x 44 columns
Clients: 32
Content types: {'keyword article': 27207, 'feedly article': 2096, 'comparison article': 697}

Base rate of decline (is_declining_label): 54.2%


## 1. My rule and its reason codes

### The rule in plain words

**A page is worth prioritising for refresh if it attracts meaningful search volume, its content is getting stale (not updated in 91+ days), AND its CTR is lower than expected for where it ranks.**

Two FlyRank flag-linked signals motivate this rule:
- **Staleness** — the same signal that drives FlyRank's refresh flags (`days_since_last_update` thresholds).
- **CTR-vs-position** — the signal behind FlyRank's CTR-fix logic: a page ranking in a visible position but failing to convert impressions to clicks.

### Reason codes this rule outputs

| Reason code | Meaning |
|---|---|
| `stale_and_low_ctr` | 91+ days since update AND CTR below position-tier median — top priority |
| `stale_only` | 91+ days since update, CTR is acceptable |
| `low_ctr_only` | CTR is low for its position but recently updated |
| `no_signal` | Neither condition triggers |

### Signal check plan

1. **Signal 1 — Staleness** (`freshness_tier`) vs. decline — flag-linked (refresh flags)
2. **Signal 2 — CTR vs. position tier** — flag-linked (CTR-fix logic)

At least one signal must be linked to a real FlyRank flag — both are.

### Signal 1 — Staleness (`days_since_last_update`) vs. decline

In [2]:
# Signal 1: freshness_tier bucket table vs. is_declining_label
# freshness_tier buckets in this dataset: 0-30 / 31-90 / 91-180 / 181+
# This is the same staleness threshold that drives FlyRank refresh flags.

freshness_order = ['0-30', '31-90', '91-180', '181+']
sig1 = (
    df[df['freshness_tier'].isin(freshness_order)]
    .groupby('freshness_tier')
    .agg(
        n=('is_declining_label', 'count'),
        decline_rate_pct=('is_declining_label', 'mean'),
        median_days=('days_since_last_update', 'median')
    )
    .reindex(freshness_order)
)
sig1['decline_rate_pct'] = (sig1['decline_rate_pct'] * 100).round(1)

n_valid = df['freshness_tier'].isin(freshness_order).sum()
print(f'Signal 1 -- Staleness bucket table  (n={n_valid:,})')
print(sig1.to_string())
print(f'\nBase rate (all rows): {base_rate:.1%}')

Signal 1 -- Staleness bucket table  (n=30,000)
                    n  decline_rate_pct  median_days
freshness_tier                                      
0-30            20480              51.1         20.0
31-90             175              58.9         41.0
91-180           9171              61.1        104.0
181+              174              47.1        211.0

Base rate (all rows): 54.2%


In [3]:
# Signal 1 verdict
low_rate  = sig1.loc['0-30',  'decline_rate_pct'] if '0-30'  in sig1.index else None
high_rate = sig1.loc['181+', 'decline_rate_pct'] if '181+' in sig1.index else None

print('Signal 1 -- VERDICT')
print(f'  Freshest pages (0-30d):  decline rate = {low_rate}%')
print(f'  Stalest pages (181+d):   decline rate = {high_rate}%')
print()

if high_rate is not None and low_rate is not None:
    diff = high_rate - low_rate
    if diff > 5:
        verdict1 = 'CONFIRMED'
        reason1 = f'Stale pages decline more ({diff:+.1f} pp above freshest tier).'
    elif diff < -5:
        verdict1 = 'OPPOSITE'
        reason1 = f'Stale pages decline LESS ({diff:+.1f} pp) -- intuition reversed.'
    else:
        verdict1 = 'MIXED'
        reason1 = f'Only {diff:+.1f} pp difference -- staleness alone is weak in this slice.'
    print(f'  VERDICT: {verdict1}')
    print(f'  Reason : {reason1}')

Signal 1 -- VERDICT
  Freshest pages (0-30d):  decline rate = 51.1%
  Stalest pages (181+d):   decline rate = 47.1%

  VERDICT: MIXED
  Reason : Only -4.0 pp difference -- staleness alone is weak in this slice.


### Signal 2 — CTR vs. position tier

In [4]:
# Signal 2: for each position tier, split pages into low-CTR vs. not-low-CTR
# then compare decline rates. This mirrors FlyRank's CTR-fix logic.
# 'Low CTR' = below the median CTR for that position tier.

position_order = ['top_3', 'page_1', 'striking', 'page_3_5']
df_pos = df[
    (df['avg_position'] > 0) &
    (df['position_tier'].isin(position_order)) &
    (df['impressions_90d'] >= 100)  # noise guard
].copy()

tier_median_ctr = df_pos.groupby('position_tier')['ctr'].median()
df_pos['low_ctr'] = df_pos.apply(
    lambda r: 1 if r['ctr'] < tier_median_ctr[r['position_tier']] else 0, axis=1
)

sig2 = (
    df_pos.groupby(['position_tier', 'low_ctr'])
    .agg(
        n=('is_declining_label', 'count'),
        decline_rate_pct=('is_declining_label', 'mean')
    )
)
sig2['decline_rate_pct'] = (sig2['decline_rate_pct'] * 100).round(1)

print(f'Signal 2 -- CTR vs. position tier  (n={len(df_pos):,} rows with impr>=100 and real position)')
print(sig2.to_string())
print(f'\nMedian CTR per position tier:')
print(tier_median_ctr.reindex(position_order).round(3).to_string())
print(f'\nBase rate (all rows): {base_rate:.1%}')

Signal 2 -- CTR vs. position tier  (n=21,127 rows with impr>=100 and real position)
                          n  decline_rate_pct
position_tier low_ctr                        
page_1        0        4371              53.5
              1        4262              68.0
page_3_5      0        3144              58.0
              1        2914              58.8
striking      0        3036              57.3
              1        2867              68.3
top_3         0         269              64.3
              1         264              87.1

Median CTR per position tier:
position_tier
top_3       0.19
page_1      0.23
striking    0.15
page_3_5    0.06

Base rate (all rows): 54.2%


In [5]:
# Signal 2 verdict
rates_low  = sig2.xs(1, level='low_ctr')['decline_rate_pct'].reindex(position_order)
rates_high = sig2.xs(0, level='low_ctr')['decline_rate_pct'].reindex(position_order)
avg_diff   = (rates_low - rates_high).mean()

print('Signal 2 -- VERDICT')
print(f'  Avg decline-rate gap (low-CTR minus high-CTR): {avg_diff:+.1f} pp')
print()
for tier in position_order:
    lo = rates_low.get(tier)
    hi = rates_high.get(tier)
    if lo is not None and hi is not None and not (np.isnan(lo) or np.isnan(hi)):
        print(f'  {tier:<12}: low-CTR={lo:.1f}%  high-CTR={hi:.1f}%  gap={lo-hi:+.1f} pp')
print()
if avg_diff > 3:
    verdict2 = 'CONFIRMED'
    reason2  = f'Low-CTR pages decline more on average ({avg_diff:+.1f} pp).'
elif avg_diff < -3:
    verdict2 = 'OPPOSITE'
    reason2  = f'Low-CTR pages decline LESS ({avg_diff:+.1f} pp) -- intuition reversed.'
else:
    verdict2 = 'MIXED'
    reason2  = f'Inconsistent across tiers (avg {avg_diff:+.1f} pp gap).'
print(f'  VERDICT: {verdict2}')
print(f'  Reason : {reason2}')

Signal 2 -- VERDICT
  Avg decline-rate gap (low-CTR minus high-CTR): +12.3 pp

  top_3       : low-CTR=87.1%  high-CTR=64.3%  gap=+22.8 pp
  page_1      : low-CTR=68.0%  high-CTR=53.5%  gap=+14.5 pp
  striking    : low-CTR=68.3%  high-CTR=57.3%  gap=+11.0 pp
  page_3_5    : low-CTR=58.8%  high-CTR=58.0%  gap=+0.8 pp

  VERDICT: CONFIRMED
  Reason : Low-CTR pages decline more on average (+12.3 pp).


## 2. Build the ranked queue (writes the CSV)

### The rule (ONE rule, hand-written, no fitted weights)

**Plain words:** A page is a refresh priority if it has visible search demand (>=300 impressions), is stale (91+ days since last update), AND has a below-median CTR for its position tier.

```
visible = impressions_90d >= 300
stale   = days_since_last_update >= 91
low_ctr = ctr < tier_median_ctr[position_tier]   (0 for no_data/deep tiers)

score = visible * stale * log1p(impressions_90d) * (1 + low_ctr)
```

`(1 + low_ctr)` doubles the score when the CTR problem is also present — two reasons to refresh beats one.

**ONE reason code:** `stale_and_low_ctr`  
**Action label:** `REFRESH_PRIORITY`

In [6]:
# Encode the rule -- no fitted weights, all thresholds hand-written
# NEVER using trend_direction, trend_pct, or any future-window column

scoring_df = df.copy()

# Visibility gate: enough search demand to matter
scoring_df['visible'] = (scoring_df['impressions_90d'] >= 300).astype(int)

# Staleness gate: content not updated in 91+ days
scoring_df['stale'] = (scoring_df['days_since_last_update'] >= 91).astype(int)

# Low-CTR flag: below position-tier median (only for visible tiers)
valid_tiers = ['top_3', 'page_1', 'striking', 'page_3_5']
tier_medians = (
    scoring_df[scoring_df['position_tier'].isin(valid_tiers)]
    .groupby('position_tier')['ctr'].median()
)
scoring_df['tier_median_ctr'] = scoring_df['position_tier'].map(tier_medians)
scoring_df['low_ctr'] = np.where(
    scoring_df['position_tier'].isin(valid_tiers) & scoring_df['tier_median_ctr'].notna(),
    (scoring_df['ctr'] < scoring_df['tier_median_ctr']).astype(int),
    0
)

# Score formula -- readable on purpose
scoring_df['score'] = (
    scoring_df['visible'] *
    scoring_df['stale'] *
    np.log1p(scoring_df['impressions_90d']) *
    (1 + scoring_df['low_ctr'])
)

# Reason code
def reason_code(row):
    v, s, l = row['visible'], row['stale'], row['low_ctr']
    if v and s and l: return 'stale_and_low_ctr'
    if v and s:       return 'stale_only'
    if v and l:       return 'low_ctr_only'
    return 'no_signal'

scoring_df['reason_code'] = scoring_df.apply(reason_code, axis=1)

# Action label
scoring_df['action'] = np.where(scoring_df['score'] > 0, 'REFRESH_PRIORITY', 'MONITOR')

# Rank highest score first
scoring_df = scoring_df.sort_values('score', ascending=False).reset_index(drop=True)
scoring_df['rank'] = scoring_df.index + 1

print(f'Total scored rows: {len(scoring_df):,}')
print(f'REFRESH_PRIORITY:  {(scoring_df["action"] == "REFRESH_PRIORITY").sum():,}')
print()
print('Reason code distribution:')
print(scoring_df['reason_code'].value_counts().to_string())

Total scored rows: 30,000
REFRESH_PRIORITY:  7,234

Reason code distribution:
reason_code
no_signal            19149
stale_only            4834
low_ctr_only          3617
stale_and_low_ctr     2400


In [7]:
# Write the ranked queue CSV
# work/**/*.csv is gitignored by design -- the notebook regenerates it on every run

OUTPUT_COLS = [
    'rank', 'content_id', 'client_id', 'content_type',
    'impressions_90d', 'avg_position', 'ctr',
    'days_since_last_update', 'freshness_tier',
    'visible', 'stale', 'low_ctr',
    'score', 'reason_code', 'action',
    'is_declining_label'   # included for evaluation only, never a model input
]

output_df = scoring_df[OUTPUT_COLS].copy()

# Resolve output dir -- works from Colab and locally
OUT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'work', 'outputs'))
os.makedirs(OUT_DIR, exist_ok=True)
OUT_PATH = os.path.join(OUT_DIR, 'baseline_action_score.csv')
output_df.to_csv(OUT_PATH, index=False)
print(f'Written: {OUT_PATH}')
print(f'Shape:   {output_df.shape}')
print()
print('Preview of top 5 rows:')
print(output_df.head().to_string(index=False))

Written: c:\Users\carlosjames\Projects\flyrank-ml-internship\work\outputs\baseline_action_score.csv
Shape:   (30000, 16)

Preview of top 5 rows:
 rank           content_id         client_id    content_type  impressions_90d  avg_position  ctr  days_since_last_update freshness_tier  visible  stale  low_ctr     score       reason_code           action  is_declining_label
    1 content_5fe46e04994d client_4e07408562 keyword article           517715           4.2 0.14                     104         91-180        1      1        1 26.314364 stale_and_low_ctr REFRESH_PRIORITY                   1
    2 content_36ff89c8214e client_19581e27de keyword article           295097           7.3 0.05                     104         91-180        1      1        1 25.190126 stale_and_low_ctr REFRESH_PRIORITY                   0
    3 content_c8e9d6ab9013 client_19581e27de keyword article           208678           9.7 0.00                     104         91-180        1      1        1 24.497105 stale_

In [8]:
# Precision@K -- this baseline is what the Week-5 model must beat

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

scores_all = output_df['score'].values
labels_all = output_df['is_declining_label'].values

p10  = precision_at_k(scores_all, labels_all, 10)
p50  = precision_at_k(scores_all, labels_all, 50)
p100 = precision_at_k(scores_all, labels_all, 100)

print('Baseline Precision@K  (Week-5 model must exceed this)')
print(f'  Base rate (random queue): {base_rate:.1%}')
print(f'  Precision@10:             {p10:.1%}')
print(f'  Precision@50:             {p50:.1%}')
print(f'  Precision@100:            {p100:.1%}')
print(f'  Gain vs. random @10:      {p10 - base_rate:+.1%}')
print(f'  Gain vs. random @50:      {p50 - base_rate:+.1%}')

# Save metrics receipt to work/outputs/*.json -- committed to git as the run receipt
metrics = {
    'assignment': 'ML-07',
    'lane': 'Refresh / Content Opportunity Scoring',
    'rule': 'stale_and_low_ctr',
    'base_rate': round(float(base_rate), 4),
    'precision_at_10':  round(float(p10),  4),
    'precision_at_50':  round(float(p50),  4),
    'precision_at_100': round(float(p100), 4),
    'n_refresh_priority': int((output_df['action'] == 'REFRESH_PRIORITY').sum()),
    'n_total': int(len(output_df))
}
metrics_path = os.path.join(OUT_DIR, 'w04_baseline_metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'\nMetrics saved: {metrics_path}')

Baseline Precision@K  (Week-5 model must exceed this)
  Base rate (random queue): 54.2%
  Precision@10:             70.0%
  Precision@50:             52.0%
  Precision@100:            53.0%
  Gain vs. random @10:      +15.8%
  Gain vs. random @50:      -2.2%

Metrics saved: c:\Users\carlosjames\Projects\flyrank-ml-internship\work\outputs\w04_baseline_metrics.json


## 3. Top-10 review

*For each of the top 10: the action, why it is there, and what would make it wrong.*

In [9]:
top10 = output_df.head(10)
review_cols = [
    'rank', 'content_type', 'impressions_90d', 'avg_position', 'ctr',
    'days_since_last_update', 'freshness_tier', 'reason_code', 'action', 'is_declining_label'
]
print('Top 10 ranked queue')
print(top10[review_cols].to_string(index=False))

Top 10 ranked queue
 rank    content_type  impressions_90d  avg_position  ctr  days_since_last_update freshness_tier       reason_code           action  is_declining_label
    1 keyword article           517715           4.2 0.14                     104         91-180 stale_and_low_ctr REFRESH_PRIORITY                   1
    2 keyword article           295097           7.3 0.05                     104         91-180 stale_and_low_ctr REFRESH_PRIORITY                   0
    3 keyword article           208678           9.7 0.00                     104         91-180 stale_and_low_ctr REFRESH_PRIORITY                   1
    4 keyword article           201111           5.7 0.11                     104         91-180 stale_and_low_ctr REFRESH_PRIORITY                   0
    5 keyword article           159590           7.8 0.06                     104         91-180 stale_and_low_ctr REFRESH_PRIORITY                   0
    6 keyword article           152467           6.5 0.13           

In [10]:
# One-line review: action / why / what would make it wrong

wrong_reasons = {
    'stale_and_low_ctr': (
        'If the page has navigational or brand intent where low CTR is expected; '
        'or if the content was silently refreshed without updating the timestamp; '
        'or if the CTR drop comes from a SERP layout change (featured snippet, ads) '
        'not from content quality decay.'
    ),
    'stale_only': (
        'If the content is genuinely evergreen and traffic is stable. '
        'Staleness alone scored MIXED in Signal 1 -- this pick rests on a weak signal.'
    ),
    'low_ctr_only': (
        'If the keyword category inherently gets low CTR at that position '
        '(e.g. informational queries dominated by knowledge panels or ads).'
    ),
    'no_signal': 'Score is 0 -- should not appear in top 10.',
}

print('Top-10 Hand Review')
print()
for _, row in top10.iterrows():
    rank  = int(row['rank'])
    impr  = int(row['impressions_90d'])
    pos   = row['avg_position']
    ctr_v = row['ctr']
    days  = int(row['days_since_last_update'])
    rc    = row['reason_code']
    label = int(row['is_declining_label'])
    ctype = row['content_type']
    act   = row['action']
    print(f'Rank {rank}: {act}  [label={label}, type={ctype}]')
    print(f'  Why here:  {impr:,} impressions, position={pos:.0f}, CTR={ctr_v:.2f}%, {days}d since update ({rc})')
    print(f'  Wrong if:  {wrong_reasons.get(rc, "n/a")}')
    print()

Top-10 Hand Review

Rank 1: REFRESH_PRIORITY  [label=1, type=keyword article]
  Why here:  517,715 impressions, position=4, CTR=0.14%, 104d since update (stale_and_low_ctr)
  Wrong if:  If the page has navigational or brand intent where low CTR is expected; or if the content was silently refreshed without updating the timestamp; or if the CTR drop comes from a SERP layout change (featured snippet, ads) not from content quality decay.

Rank 2: REFRESH_PRIORITY  [label=0, type=keyword article]
  Why here:  295,097 impressions, position=7, CTR=0.05%, 104d since update (stale_and_low_ctr)
  Wrong if:  If the page has navigational or brand intent where low CTR is expected; or if the content was silently refreshed without updating the timestamp; or if the CTR drop comes from a SERP layout change (featured snippet, ads) not from content quality decay.

Rank 3: REFRESH_PRIORITY  [label=1, type=keyword article]
  Why here:  208,678 impressions, position=10, CTR=0.00%, 104d since update (stale_a

## 4. Weak picks + leakage check

### Weak picks

The score formula rewards high search volume (via log1p) heavily, so the top of the queue is dominated by pages with large impression counts that are also stale. Some of these are NOT actually declining (label=0) — the rule flags them because they are big and old, not because they are losing traffic. These are the honest false alarms this baseline must acknowledge.

A secondary pattern: pages in `freshness_tier=91-180` score lower than `181+`, which is expected. But since Signal 1 may come back MIXED, even the 181+ tier does not guarantee a declining page. The Week-5 model needs to learn when volume is misleading.

In [11]:
# Weak picks: top-10 rows where label=0 (false alarms)
review_cols = [
    'rank', 'content_type', 'impressions_90d', 'avg_position', 'ctr',
    'days_since_last_update', 'freshness_tier', 'reason_code', 'action', 'is_declining_label'
]
false_alarms = top10[top10['is_declining_label'] == 0]
print(f'Weak picks in top 10 (label=0 / not actually declining): {len(false_alarms)}')
if len(false_alarms) > 0:
    print(false_alarms[review_cols].to_string(index=False))
print()

# False alarm rate in top 50
top50 = output_df.head(50)
fa50  = (top50['is_declining_label'] == 0).sum()
print(f'False alarms in top 50: {fa50} / 50  ({fa50/50:.1%})')
print(f'Correct picks in top 50: {50-fa50} / 50  ({(50-fa50)/50:.1%})')

Weak picks in top 10 (label=0 / not actually declining): 3
 rank    content_type  impressions_90d  avg_position  ctr  days_since_last_update freshness_tier       reason_code           action  is_declining_label
    2 keyword article           295097           7.3 0.05                     104         91-180 stale_and_low_ctr REFRESH_PRIORITY                   0
    4 keyword article           201111           5.7 0.11                     104         91-180 stale_and_low_ctr REFRESH_PRIORITY                   0
    5 keyword article           159590           7.8 0.06                     104         91-180 stale_and_low_ctr REFRESH_PRIORITY                   0

False alarms in top 50: 24 / 50  (48.0%)
Correct picks in top 50: 26 / 50  (52.0%)


In [12]:
# Leakage check -- confirm no forbidden (label-derived or future-window) columns
# are used as features in the scoring rule.

FORBIDDEN = [
    'trend_direction',    # label source
    'trend_pct',          # label source
    'is_declining_label', # the label itself
    'impressions_last_30d',
    'clicks_last_30d',
    'sessions_last_30d',
]

FEATURES_USED = [
    'impressions_90d',         # 90-day total -- past window
    'days_since_last_update',  # content metadata -- past
    'avg_position',            # 90-day avg position -- past
    'ctr',                     # 90-day CTR -- past
    'freshness_tier',          # derived from days_since_last_update -- past
    'position_tier',           # derived from avg_position -- past
]

print('Leakage check')
print('=' * 50)
all_clean = True
for col in FORBIDDEN:
    if col in FEATURES_USED:
        print(f'  LEAK DETECTED: {col}')
        all_clean = False
    else:
        print(f'  OK (not used as feature): {col}')
print()
if all_clean:
    print('CLEAN -- no forbidden columns in the scoring rule.')
else:
    print('WARNING -- leakage found. Fix before Week 5.')
print()
print('Features used in the rule:')
for f in FEATURES_USED:
    print(f'  {f}')

Leakage check
  OK (not used as feature): trend_direction
  OK (not used as feature): trend_pct
  OK (not used as feature): is_declining_label
  OK (not used as feature): impressions_last_30d
  OK (not used as feature): clicks_last_30d
  OK (not used as feature): sessions_last_30d

CLEAN -- no forbidden columns in the scoring rule.

Features used in the rule:
  impressions_90d
  days_since_last_update
  avg_position
  ctr
  freshness_tier
  position_tier


## Self-check

Before you submit, confirm each line honestly:

- [x] **Two signal checks** with bucket tables and n printed — Signal 1 (staleness/freshness_tier, flag-linked to refresh flags) and Signal 2 (CTR vs. position tier, flag-linked to CTR-fix logic), each with a one-word verdict
- [x] **ONE rule encoded** — score using `visible * stale * log1p(impressions_90d) * (1 + low_ctr)`, one reason code (`stale_and_low_ctr`), one action label (`REFRESH_PRIORITY`)
- [x] **Ranked queue written** — `work/outputs/baseline_action_score.csv` from this notebook
- [x] **Top-10 review** — for each row: action, why it is there, what would make it wrong
- [x] **Weak picks identified** — false alarms (label=0 in top 10) explicitly called out
- [x] **Leakage check** — `trend_direction`, `trend_pct`, and future-window columns confirmed absent from features
- [ ] Notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] Claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to repo under `work/notebooks/` — then submit repo URL on the card. Done.